In [10]:
!pip install gdown
import gdown

In [11]:
# https://drive.google.com/file/d/1VvAI_EmjfPl-MHsyEOlLc9rE2OZ4jRtY/view?usp=sharing
file_id = '1VvAI_EmjfPl-MHsyEOlLc9rE2OZ4jRtY'
gdown_url = f'https://drive.google.com/uc?id={file_id}'
output_path = '../dataset/'

In [12]:
gdown.download(gdown_url, output_path, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1VvAI_EmjfPl-MHsyEOlLc9rE2OZ4jRtY
From (redirected): https://drive.google.com/uc?id=1VvAI_EmjfPl-MHsyEOlLc9rE2OZ4jRtY&confirm=t&uuid=f82712a1-ec1c-43a2-9fc9-e8c1a5d2c2b5
To: /workspace/FSL-100-Recognizer/dataset/clips.zip
100%|██████████| 379M/379M [00:05<00:00, 72.2MB/s] 


'../dataset/clips.zip'

In [24]:
# !unzip ../dataset/clips.zip -d ../dataset/

In [13]:
# #%%
# import os
# import cv2
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader, random_split
# import pandas as pd
# import numpy as np
# # CONFIGURATION

# # -------- Dataset Class --------
# class VideoDataset(Dataset):
#     def __init__(self, video_folder, csv_file, num_frames=16, frame_size=(112, 112)):
#         self.video_folder = video_folder
#         self.labels_df = pd.read_csv(csv_file)
#         self.num_frames = num_frames
#         self.frame_size = frame_size

#     def __len__(self):
#         return len(self.labels_df)

#     def __getitem__(self, idx):
#         row = self.labels_df.iloc[idx]
#         video_path = os.path.join(self.video_folder, row['filename'])
#         label = int(row['label'])

#         frames = self._load_video_frames(video_path)
#         return frames, label

#     def _load_video_frames(self, video_path):
#         cap = cv2.VideoCapture(video_path)
#         total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

#         # Choose frame indices uniformly
#         frame_indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)

#         frames = []
#         for i in range(total_frames):
#             ret, frame = cap.read()
#             if not ret:
#                 break
#             if i in frame_indices:
#                 frame = cv2.resize(frame, self.frame_size)
#                 frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#                 frame = frame / 255.0  # normalize
#                 frame = np.transpose(frame, (2, 0, 1))  # CxHxW
#                 frames.append(frame)

#         cap.release()

#         # If video was too short, pad with zeros
#         while len(frames) < self.num_frames:
#             frames.append(np.zeros((3, *self.frame_size)))

#         frames = np.stack(frames)  # Shape: [num_frames, 3, H, W]
#         return torch.tensor(frames, dtype=torch.float32)

# # -------- GRU Model --------
# class VideoGRUModel(nn.Module):
#     def __init__(self, input_size, hidden_size, num_layers, num_classes):
#         super(VideoGRUModel, self).__init__()
#         self.hidden_size = hidden_size
#         self.num_layers = num_layers

#         self.feature_dim = input_size[0] * input_size[1] * input_size[2]

#         self.gru = nn.GRU(
#             input_size=self.feature_dim,
#             hidden_size=hidden_size,
#             num_layers=num_layers,
#             batch_first=True
#         )
#         self.fc = nn.Linear(hidden_size, num_classes)

#     def forward(self, x):
#         batch_size, seq_len, C, H, W = x.shape
#         x = x.view(batch_size, seq_len, -1)  # flatten each frame
#         out, _ = self.gru(x)
#         out = out[:, -1, :]   # last hidden state
#         out = self.fc(out)
#         return out

# # -------- Train Function --------
# def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=10, device='cpu'):
#     model.to(device)
#     for epoch in range(num_epochs):
#         model.train()
#         running_loss = 0.0
#         correct = 0
#         total = 0
#         for inputs, labels in train_loader:
#             inputs = inputs.to(device)
#             labels = labels.to(device)

#             outputs = model(inputs)
#             loss = criterion(outputs, labels)

#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()

#             running_loss += loss.item()

#             _, predicted = outputs.max(1)
#             total += labels.size(0)
#             correct += predicted.eq(labels).sum().item()

#         train_acc = 100. * correct / total
#         print(f"\nEpoch [{epoch+1}/{num_epochs}] Train Loss: {running_loss/len(train_loader):.4f} "
#               f"Train Acc: {train_acc:.2f}%")

#         # Evaluate on test set
#         test_acc = evaluate_model(model, test_loader, device)
#         print(f"Test Acc: {test_acc:.2f}%\n")

# # -------- Evaluation Function --------
# def evaluate_model(model, dataloader, device='cpu'):
#     model.eval()
#     correct = 0
#     total = 0
#     with torch.no_grad():
#         for inputs, labels in dataloader:
#             inputs = inputs.to(device)
#             labels = labels.to(device)

#             outputs = model(inputs)
#             _, predicted = outputs.max(1)
#             total += labels.size(0)
#             correct += predicted.eq(labels).sum().item()

#     acc = 100. * correct / total
#     return acc

# # -------- Main --------
# if __name__ == '__main__':
#     # Parameters
#     # video_folder = '/path/to/videos'
#     # csv_file = '/path/to/labels.csv'

#     video_folder = 'workspace/clips/'  # Change this
#     csv_file = 'spliced_train.csv'  # Change this
#     num_frames = 16
#     frame_size = (112, 112)
#     batch_size = 4
#     hidden_size = 128
#     num_layers = 1
#     num_classes = 3
#     num_epochs = 30
#     learning_rate = 0.001
#     test_split_ratio = 0.2  # 20% for test


#     # PREPROCESSING =======================================================
#     # Full dataset 
#     full_dataset = VideoDataset(video_folder, csv_file, num_frames, frame_size)

#     # Split into train/test
#     test_size = int(len(full_dataset) * test_split_ratio)
#     train_size = len(full_dataset) - test_size
#     train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

#     # Dataloaders
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#     test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

#     # PREPROCESSING =======================================================

#     # TRAINING ============================================================
#     # Model
#     model = VideoGRUModel(input_size=(3, *frame_size), hidden_size=hidden_size,
#                           num_layers=num_layers, num_classes=num_classes)

#     # Loss & Optimizer
#     criterion = nn.CrossEntropyLoss()
#     optimizer = optim.Adam(model.parameters(), lr=learning_rate)

#     # TRAINING ============================================================

#     # Train & Evaluate
#     train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=num_epochs,
#                 device='cuda' if torch.cuda.is_available() else 'cpu')


In [16]:
import os
import csv
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import mediapipe as mp
# --------------------
# Dataset Definition
# --------------------
class HandSignDataset(Dataset):
    def __init__(self, videos_dir, csv_path, seq_length=50, transform=None):
        self.videos_dir = videos_dir
        self.seq_length = seq_length
        self.transform = transform

        # Read CSV mapping video filenames to labels
        self.samples = []  # list of (video_path, label)
        with open(csv_path, 'r') as f:
            reader = csv.reader(f)
            header = next(reader)  # assume header: filename,label
            for row in reader:
                fname, label = row
                path = os.path.join(videos_dir, fname)
                self.samples.append((path, int(label)))

        # Initialize MediaPipe hand detector
        self.mp_hands = mp.solutions.hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        cap = cv2.VideoCapture(video_path)
        landmarks_seq = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            # Convert BGR to RGB
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.mp_hands.process(rgb)

            if results.multi_hand_landmarks:
                hand_landmarks = results.multi_hand_landmarks[0]
                # Extract 21 (x,y,z) coords
                coords = []
                for lm in hand_landmarks.landmark:
                    coords.extend([lm.x, lm.y, lm.z])
                landmarks_seq.append(coords)

            # stop if exceed sequence length
            if len(landmarks_seq) >= self.seq_length:
                break

        cap.release()

        # Pad or truncate sequence to fixed length
        seq = torch.zeros(self.seq_length, 21 * 3)
        for i, frame_coords in enumerate(landmarks_seq[:self.seq_length]):
            seq[i] = torch.tensor(frame_coords)

        if self.transform:
            seq = self.transform(seq)

        return seq, label

# --------------------
# Model Definition (GRU-based)
# --------------------
class GRUClassifier(nn.Module):
    def __init__(self, input_size=63, hidden_size=128, num_layers=2, num_classes=10):
        super(GRUClassifier, self).__init__()
        self.hidden_size = hidden_size  # CHANGE
        self.num_layers = num_layers    # CHANGE
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),  # CHANGE
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, hidden = self.gru(x)  # out: (batch, seq_len, hidden_size*2)
        # hidden shape: (num_layers*2, batch, hidden_size)
        # CHANGE: extract last-layer forward and backward hidden states
        h_forward = hidden[-2]    # (batch, hidden_size)  CHANGE
        h_backward = hidden[-1]   # (batch, hidden_size)  CHANGE
        last = torch.cat((h_forward, h_backward), dim=1)  # (batch, hidden_size*2)  CHANGE
        return self.classifier(last)

# --------------------
# Training Loop
# --------------------
def train_model(videos_dir, csv_path, num_classes,
                batch_size=8, lr=1e-3, epochs=20, seq_length=50, device='cuda'):
    dataset = HandSignDataset(videos_dir, csv_path, seq_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = GRUClassifier(input_size=63, hidden_size=128, num_layers=2, num_classes=num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for sequences, labels in dataloader:
            sequences = sequences.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * sequences.size(0)
            _, preds = torch.max(outputs, 1)
            # print(f'PREDs:  {str(preds)}')
            # print(f'Labels:  {str(labels)}')
            # print(labels)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_acc = correct / total
        print(f"Epoch {epoch}/{epochs} - Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}")

    # Save the trained model
    torch.save(model.state_dict(), 'gru_hand_sign_model.pth')
    print("Training complete. Model saved to gru_hand_sign_model.pth")

# --------------------
# Single-Sample Inference
# --------------------
def infer_one_sample(videos_dir, csv_path, model_path,
                     sample_idx=0, seq_length=50, device='cuda'):
    # Load dataset and sample
    dataset = HandSignDataset(videos_dir, csv_path, seq_length)
    seq, true_label = dataset[sample_idx]

    # Load model
    model = GRUClassifier(input_size=63, hidden_size=128, num_layers=2, num_classes=max([lbl for _, lbl in dataset.samples])+1)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()

    # Prepare input
    seq = seq.unsqueeze(0).to(device)  # shape: (1, seq_length, 63)

    # Inference
    with torch.no_grad():
        outputs = model(seq)
        probs = torch.softmax(outputs, dim=1)
        pred_label = torch.argmax(probs, dim=1).item()

    print(f"Sample index: {sample_idx}")
    print(f"Expected label: {true_label}")
    print(f"Predicted label: {pred_label}")
    print(f"Probabilities: {probs.cpu().numpy()}")
    

In [17]:
# !pip install mediapipe

In [18]:
# VIDEOS_DIR = 'data/videos'
# CSV_PATH = 'data/spliced_train.csv'
# NUM_CLASSES = 10  # adjust to your dataset
# VIDEOS_DIR = 'workspace/clips/'  # Change this
# CSV_PATH = 'spliced_train.csv'  # Change this

# train_model(
#     videos_dir=VIDEOS_DIR,
#     csv_path=CSV_PATH,
#     num_classes=NUM_CLASSES,
#     batch_size=4,
#     lr=1e-3,
#     epochs=30,
#     seq_length=50,
#     device='cuda' if torch.cuda.is_available() else 'cpu'
# )



In [19]:
# infer_one_sample(
#     videos_dir=VIDEOS_DIR,
#     csv_path=CSV_PATH,
#     model_path='gru_hand_sign_model.pth',
#     sample_idx=0,
#     seq_length=50,
#     device='cuda' if torch.cuda.is_available() else 'cpu'
# )


In [20]:
VIDEOS_DIR = 'workspace/clips/'  # Change this
CSV_PATH = 'spliced_train.csv'  # Change this
seq_length = 50
batch_size= 4

dataset = HandSignDataset(VIDEOS_DIR,CSV_PATH)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

I0000 00:00:1750481126.873769   20729 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1750481126.933125   31245 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 555.58.02), renderer: NVIDIA GeForce RTX 3060/PCIe/SSE2
W0000 00:00:1750481126.965183   31195 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1750481126.989892   31217 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [21]:
num_classes = 105
device='cuda' if torch.cuda.is_available() else 'cpu'
model = GRUClassifier(input_size=63, hidden_size=128, num_layers=2, num_classes=num_classes)
model = model.to(device)

In [22]:
lr=1e-3

In [23]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
batch_size=8 
lr=1e-3
epochs=20
seq_length=50

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for sequences, labels in dataloader:
        sequences = sequences.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * sequences.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch}/{epochs} - Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}")

# Save the trained model
torch.save(model.state_dict(), 'gru_hand_sign_model.pth')
print("Training complete. Model saved to gru_hand_sign_model.pth")

Epoch 1/20 - Loss: 4.6642  Acc: 0.0041
Epoch 2/20 - Loss: 4.6584  Acc: 0.0094
Epoch 3/20 - Loss: 4.6580  Acc: 0.0088
Epoch 4/20 - Loss: 4.6576  Acc: 0.0059
Epoch 5/20 - Loss: 4.6574  Acc: 0.0082
Epoch 6/20 - Loss: 4.6573  Acc: 0.0070
Epoch 7/20 - Loss: 4.6570  Acc: 0.0065
Epoch 8/20 - Loss: 4.6568  Acc: 0.0053
Epoch 9/20 - Loss: 4.6566  Acc: 0.0059
Epoch 10/20 - Loss: 4.6566  Acc: 0.0070
Epoch 11/20 - Loss: 4.6564  Acc: 0.0094
Epoch 12/20 - Loss: 4.6563  Acc: 0.0076
Epoch 13/20 - Loss: 4.6563  Acc: 0.0059
Epoch 14/20 - Loss: 4.6562  Acc: 0.0059
Epoch 15/20 - Loss: 4.6562  Acc: 0.0070
Epoch 16/20 - Loss: 4.6560  Acc: 0.0076
Epoch 17/20 - Loss: 4.6561  Acc: 0.0065
Epoch 18/20 - Loss: 4.6560  Acc: 0.0041
Epoch 19/20 - Loss: 4.6560  Acc: 0.0053
Epoch 20/20 - Loss: 4.6560  Acc: 0.0065
Training complete. Model saved to gru_hand_sign_model.pth


In [24]:
model

GRUClassifier(
  (gru): GRU(63, 128, num_layers=2, batch_first=True)
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=64, out_features=105, bias=True)
  )
)

In [25]:
# Load dataset and sample
count = 0
dataset = HandSignDataset(VIDEOS_DIR, CSV_PATH, seq_length)
for sample_idx in range(0,17):

    seq, true_label = dataset[sample_idx]
    
    # Load model
    # model = GRUClassifier(input_size=63, hidden_size=128, num_layers=2, num_classes=max([lbl for _, lbl in dataset.samples])+1)
    # model.load_state_dict(torch.load(model_path, map_location=device))
    # model.to(device).eval()
    
    # Prepare input
    seq = seq.unsqueeze(0).to(device)  # shape: (1, seq_length, 63)
    
    # Inference
    with torch.no_grad():
        outputs = model(seq)
        probs = torch.softmax(outputs, dim=1)
        pred_label = torch.argmax(probs, dim=1).item()
    
    print(f"Sample index: {sample_idx}")
    print(f"Expected label: {true_label}")
    print(f"Predicted label: {pred_label}")
    print(f"Probabilities: {probs.cpu().numpy()}")
    if pred_label == true_label:
        count +=1

Sample index: 0
Expected label: 66
Predicted label: 43
Probabilities: [[0.00938808 0.00978343 0.00928    0.0093875  0.0096438  0.00937929
  0.00913386 0.00983813 0.00942444 0.00912631 0.00931784 0.00937294
  0.00986821 0.00998092 0.0098043  0.00957063 0.00923842 0.00947057
  0.0097059  0.00955074 0.00975356 0.00956463 0.00963053 0.00939945
  0.01006578 0.00996952 0.00959124 0.00878516 0.00939993 0.00930232
  0.00924233 0.00932836 0.00941282 0.01012243 0.00983558 0.0094013
  0.00854583 0.00939712 0.00952114 0.00984217 0.00930823 0.00936977
  0.00953513 0.01013186 0.00931189 0.0096655  0.00987391 0.00971076
  0.00953134 0.01011505 0.00947569 0.00958337 0.00935483 0.00932922
  0.00942505 0.00971753 0.00959826 0.00956463 0.00957666 0.00926038
  0.00995346 0.00928019 0.0100541  0.00941952 0.00984919 0.00956049
  0.00952781 0.00951783 0.00943918 0.00932624 0.00962039 0.00967719
  0.0094359  0.00952425 0.00942432 0.00980617 0.00930764 0.00931974
  0.01011095 0.00928777 0.00955004 0.00975215 0

I0000 00:00:1750481178.495752   20729 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1750481178.551561   31322 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 555.58.02), renderer: NVIDIA GeForce RTX 3060/PCIe/SSE2
W0000 00:00:1750481178.583846   31273 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [26]:
count  /51

0.0

W0000 00:00:1750481178.610171   31299 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
